
# Grain Robustness Q1 — Notebook 05
## Entrenamiento final, calibración y evaluación base congelada

Este notebook utiliza dos entradas:

1. `NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip`
2. `NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip`

### Objetivo

Ajustar las 32 configuraciones ya seleccionadas:

- 2 datasets;
- 4 representaciones;
- 4 algoritmos;

utilizando exclusivamente la partición de entrenamiento, ajustar una
temperatura escalar con la partición de calibración y generar la evaluación
base en la partición de prueba congelada.

### Salidas principales

- modelos finales serializados;
- orden de clases y orden de características;
- temperatura ajustada en calibración;
- predicciones nativas y calibradas;
- métricas globales y por clase;
- matrices de confusión en formato tabular;
- figuras de desempeño base y calibración;
- manifiesto SHA-256;
- paquete ZIP para el Notebook 06.

### Reglas metodológicas

- No se vuelven a seleccionar hiperparámetros.
- No se selecciona un algoritmo ganador.
- No se utiliza la prueba para ajustar temperatura ni umbrales.
- La temperatura se estima solo con `calibration`.
- La escala base es `s = 1.00`.
- El test se utiliza únicamente para la matriz de evaluaciones ya congelada.
- La calibración por temperatura no cambia el `argmax`; por ello, las métricas
  de discriminación deben coincidir antes y después de calibrar.
- Utilizar **CPU**. No se requiere GPU.

### Tiempo estimado

Entre 5 y 25 minutos en Colab CPU. La copia de modelos a Drive puede tardar
algunos minutos adicionales.


## 1. Preparar Colab

In [ ]:
!pip -q install pyarrow catboost joblib scipy


In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## 2. Importaciones y configuración

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import platform
import re
import shutil
import sys
import time
import traceback
import zipfile
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.special import softmax

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    matthews_corrcoef,
    log_loss,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

RANDOM_SEED = 20260803
N_ECE_BINS = 15
N_ADAPTIVE_BINS = 10

LOCAL_ROOT = Path('/content/Grain_Robustness_Q1_notebook05')
INPUT_DIR = LOCAL_ROOT / '00_input'
EXTRACT03_DIR = LOCAL_ROOT / '01_extracted_notebook03'
EXTRACT04_DIR = LOCAL_ROOT / '02_extracted_notebook04'
OUTPUT_DIR = LOCAL_ROOT / '03_outputs'

MODEL_DIR = OUTPUT_DIR / '05_models'
PREDICTION_DIR = OUTPUT_DIR / '06_predictions' / '05_BASELINE'
RESULTS_DIR = OUTPUT_DIR / '07_results' / '05_BASELINE_CALIBRATION'
FIGURES_DIR = OUTPUT_DIR / '08_figures' / '05_BASELINE_CALIBRATION'
TABLES_DIR = OUTPUT_DIR / '09_tables' / '05_BASELINE_CALIBRATION'
PROTOCOL_DIR = OUTPUT_DIR / '03_protocol'

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Grain_Robustness_Q1')

for directory in [
    INPUT_DIR,
    EXTRACT03_DIR,
    EXTRACT04_DIR,
    MODEL_DIR,
    PREDICTION_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    PROTOCOL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

DATASETS = ['dry_bean', 'iniap_rice']
REPRESENTATIONS = [
    'R1_full_native',
    'R2_measured_core',
    'R3_invariant_core',
    'R4_hybrid_log_area',
]
ALGORITHMS = [
    'multinomial_logistic_regression',
    'rbf_svm',
    'extra_trees',
    'catboost',
]

METADATA_COLUMNS = {
    'dataset_id',
    'record_id',
    'source_excel_row',
    'feature_fingerprint_sha256',
    'full_row_fingerprint_sha256',
    'duplicate_group_id',
    'exact_duplicate_count',
    'class_label',
    'fold_id',
    'partition',
}

print('Espacio local:', LOCAL_ROOT)
print('Destino Drive:', DRIVE_PROJECT_ROOT)
print('CPU disponibles:', os.cpu_count())
print('Python:', sys.version.split()[0])


## 3. Localizar o cargar los dos ZIP

In [ ]:

from google.colab import files


def locate_or_upload(expected_filename, candidate_paths):
    located = next(
        (path for path in candidate_paths if path.exists()),
        None,
    )

    if located is not None:
        print('Encontrado:', located)
        return located

    print(f'Selecciona {expected_filename}')
    uploaded = files.upload()

    if expected_filename in uploaded:
        return Path('/content') / expected_filename

    uploaded_names = list(uploaded.keys())
    matching = [
        name for name in uploaded_names
        if Path(name).name == expected_filename
    ]

    if len(matching) != 1:
        raise RuntimeError(
            f'Debes cargar exactamente {expected_filename}. '
            f'Archivos recibidos: {uploaded_names}'
        )

    return Path('/content') / matching[0]


zip03_source = locate_or_upload(
    'NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip',
    [
        Path('/content/NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'),
        (
            DRIVE_PROJECT_ROOT / '07_results'
            / 'NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'
        ),
        (
            DRIVE_PROJECT_ROOT
            / 'NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'
        ),
    ],
)

zip04_source = locate_or_upload(
    'NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip',
    [
        Path('/content/NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip'),
        (
            DRIVE_PROJECT_ROOT / '07_results'
            / 'NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip'
        ),
        (
            DRIVE_PROJECT_ROOT
            / 'NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip'
        ),
    ],
)

zip03_local = INPUT_DIR / 'NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip'
zip04_local = INPUT_DIR / 'NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip'

shutil.copy2(zip03_source, zip03_local)
shutil.copy2(zip04_source, zip04_local)

print('Notebook 03 ZIP MB:', round(zip03_local.stat().st_size / 1_000_000, 3))
print('Notebook 04 ZIP MB:', round(zip04_local.stat().st_size / 1_000_000, 3))


## 4. Extraer y validar las entradas

In [ ]:

for directory in [EXTRACT03_DIR, EXTRACT04_DIR]:
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip03_local, 'r') as archive:
    archive.extractall(EXTRACT03_DIR)

with zipfile.ZipFile(zip04_local, 'r') as archive:
    archive.extractall(EXTRACT04_DIR)

split_validation_path = (
    EXTRACT03_DIR / '07_results' / '03_FROZEN_SPLITS'
    / '06_final_split_validation.csv'
)
leakage_path = (
    EXTRACT03_DIR / '07_results' / '03_FROZEN_SPLITS'
    / '02_split_leakage_audit.csv'
)
selection_validation_path = (
    EXTRACT04_DIR / '07_results' / '04_MODEL_SELECTION'
    / '05_final_model_selection_validation.csv'
)
selected_csv_path = (
    EXTRACT04_DIR / '07_results' / '04_MODEL_SELECTION'
    / '03_selected_model_configurations.csv'
)
selected_json_path = (
    EXTRACT04_DIR / '03_protocol'
    / 'selected_model_configurations_v1.json'
)

required = [
    split_validation_path,
    leakage_path,
    selection_validation_path,
    selected_csv_path,
    selected_json_path,
]

for dataset_id in DATASETS:
    for representation in REPRESENTATIONS:
        required.append(
            EXTRACT03_DIR / '05_frozen_splits'
            / dataset_id
            / f'{representation}_with_split.parquet'
        )

missing = [str(path) for path in required if not path.exists()]

if missing:
    raise FileNotFoundError(
        'Faltan archivos requeridos:\n' + '\n'.join(missing)
    )

split_validation = pd.read_csv(split_validation_path)
leakage_audit = pd.read_csv(leakage_path)
selection_validation = pd.read_csv(selection_validation_path)
selected_configurations = pd.read_csv(selected_csv_path)

if not split_validation['passed'].all():
    raise RuntimeError('La validación del Notebook 03 falló.')

if not selection_validation['passed'].all():
    raise RuntimeError('La validación del Notebook 04 falló.')

leakage_numeric = [
    column for column in leakage_audit.columns
    if column != 'dataset_id'
]

if leakage_audit[leakage_numeric].to_numpy().sum() != 0:
    raise RuntimeError('La entrada contiene fuga entre particiones.')

expected_selected_rows = (
    len(DATASETS)
    * len(REPRESENTATIONS)
    * len(ALGORITHMS)
)

if len(selected_configurations) != expected_selected_rows:
    raise RuntimeError(
        f'Se esperaban {expected_selected_rows} configuraciones; '
        f'se encontraron {len(selected_configurations)}.'
    )

print('Entradas de los Notebooks 03 y 04 validadas correctamente.')
display(split_validation)
display(selection_validation)


## 5. Cargar datos y detectar predictores

In [ ]:

datasets = {}
feature_columns = {}

for dataset_id in DATASETS:
    datasets[dataset_id] = {}
    feature_columns[dataset_id] = {}
    reference_keys = None

    for representation in REPRESENTATIONS:
        path = (
            EXTRACT03_DIR / '05_frozen_splits'
            / dataset_id
            / f'{representation}_with_split.parquet'
        )

        dataframe = pd.read_parquet(path)
        dataframe = dataframe.sort_values(
            'record_id'
        ).reset_index(drop=True)

        keys = dataframe[
            [
                'record_id',
                'class_label',
                'partition',
                'duplicate_group_id',
            ]
        ].copy()

        if reference_keys is None:
            reference_keys = keys
        elif not reference_keys.equals(keys):
            raise RuntimeError(
                f'Desalineación en {dataset_id}, {representation}.'
            )

        features = [
            column for column in dataframe.columns
            if column not in METADATA_COLUMNS
        ]

        if dataframe[features].isna().sum().sum() != 0:
            raise RuntimeError(
                f'Valores faltantes en {dataset_id}, {representation}.'
            )

        datasets[dataset_id][representation] = dataframe
        feature_columns[dataset_id][representation] = features

    partition_counts = (
        reference_keys['partition'].value_counts().to_dict()
    )
    print(dataset_id, partition_counts)


## 6. Funciones de modelos, probabilidades y calibración

In [ ]:

def build_estimator(algorithm, parameters, random_seed):
    if algorithm == 'multinomial_logistic_regression':
        return Pipeline([
            ('scaler', StandardScaler()),
            (
                'model',
                LogisticRegression(
                    C=float(parameters['C']),
                    solver='lbfgs',
                    max_iter=5000,
                    random_state=random_seed,
                ),
            ),
        ])

    if algorithm == 'rbf_svm':
        return Pipeline([
            ('scaler', StandardScaler()),
            (
                'model',
                SVC(
                    C=float(parameters['C']),
                    gamma=parameters['gamma'],
                    kernel='rbf',
                    probability=True,
                    random_state=random_seed,
                    cache_size=2000,
                ),
            ),
        ])

    if algorithm == 'extra_trees':
        return ExtraTreesClassifier(
            n_estimators=300,
            max_features=parameters['max_features'],
            min_samples_leaf=int(parameters['min_samples_leaf']),
            random_state=random_seed,
            n_jobs=-1,
        )

    if algorithm == 'catboost':
        return CatBoostClassifier(
            iterations=400,
            depth=int(parameters['depth']),
            learning_rate=float(parameters['learning_rate']),
            loss_function='MultiClass',
            random_seed=random_seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

    raise ValueError(f'Algoritmo desconocido: {algorithm}')


def aligned_probabilities(estimator, x, class_order):
    probabilities = np.asarray(
        estimator.predict_proba(x),
        dtype=float,
    )
    estimator_classes = np.asarray(
        estimator.classes_,
        dtype=str,
    )

    aligned = np.zeros(
        (len(x), len(class_order)),
        dtype=float,
    )

    destination = {
        class_label: class_index
        for class_index, class_label in enumerate(class_order)
    }

    for source_index, class_label in enumerate(estimator_classes):
        aligned[:, destination[class_label]] = (
            probabilities[:, source_index]
        )

    row_sums = aligned.sum(axis=1, keepdims=True)
    aligned = aligned / row_sums
    return aligned


def probability_temperature_scale(probabilities, temperature):
    clipped = np.clip(
        probabilities,
        1e-15,
        1.0,
    )
    log_probabilities = np.log(clipped)
    return softmax(
        log_probabilities / float(temperature),
        axis=1,
    )


def fit_temperature(probabilities, y_true_indices):
    probabilities = np.asarray(probabilities, dtype=float)
    y_true_indices = np.asarray(y_true_indices, dtype=int)

    def objective(log_temperature):
        temperature = np.exp(log_temperature)
        calibrated = probability_temperature_scale(
            probabilities,
            temperature,
        )
        selected = calibrated[
            np.arange(len(y_true_indices)),
            y_true_indices,
        ]
        return float(
            -np.mean(
                np.log(
                    np.clip(selected, 1e-15, 1.0)
                )
            )
        )

    result = minimize_scalar(
        objective,
        bounds=(np.log(0.05), np.log(10.0)),
        method='bounded',
        options={'xatol': 1e-7},
    )

    if not result.success:
        raise RuntimeError(
            f'Falló el ajuste de temperatura: {result.message}'
        )

    temperature = float(np.exp(result.x))
    return temperature, float(result.fun)


def expected_calibration_error(
    y_true_indices,
    probabilities,
    n_bins=15,
):
    confidence = probabilities.max(axis=1)
    prediction = probabilities.argmax(axis=1)
    correct = prediction == y_true_indices
    edges = np.linspace(0.0, 1.0, n_bins + 1)

    value = 0.0

    for bin_index in range(n_bins):
        if bin_index == n_bins - 1:
            mask = (
                (confidence >= edges[bin_index])
                & (confidence <= edges[bin_index + 1])
            )
        else:
            mask = (
                (confidence >= edges[bin_index])
                & (confidence < edges[bin_index + 1])
            )

        if mask.any():
            value += mask.mean() * abs(
                correct[mask].mean()
                - confidence[mask].mean()
            )

    return float(value)


def adaptive_ece(
    y_true_indices,
    probabilities,
    n_bins=10,
):
    confidence = probabilities.max(axis=1)
    prediction = probabilities.argmax(axis=1)
    correct = (prediction == y_true_indices).astype(float)

    order = np.argsort(confidence)
    bins = np.array_split(order, n_bins)

    value = 0.0
    n = len(confidence)

    for indices in bins:
        if len(indices) == 0:
            continue

        value += (
            len(indices) / n
        ) * abs(
            correct[indices].mean()
            - confidence[indices].mean()
        )

    return float(value)


def multiclass_brier(y_true_indices, probabilities):
    one_hot = np.zeros_like(probabilities)
    one_hot[
        np.arange(len(y_true_indices)),
        y_true_indices,
    ] = 1.0

    return float(
        np.mean(
            np.sum(
                (probabilities - one_hot) ** 2,
                axis=1,
            )
        )
    )


def compute_global_metrics(
    y_true_labels,
    probabilities,
    class_order,
):
    class_to_index = {
        class_label: index
        for index, class_label in enumerate(class_order)
    }
    y_true_indices = np.array(
        [
            class_to_index[class_label]
            for class_label in y_true_labels
        ],
        dtype=int,
    )
    prediction_indices = probabilities.argmax(axis=1)
    prediction_labels = np.asarray(class_order)[prediction_indices]

    y_binary = label_binarize(
        y_true_labels,
        classes=class_order,
    )

    if len(class_order) == 2:
        y_binary = np.column_stack(
            [1 - y_binary, y_binary]
        )

    return {
        'accuracy': accuracy_score(
            y_true_labels,
            prediction_labels,
        ),
        'balanced_accuracy': balanced_accuracy_score(
            y_true_labels,
            prediction_labels,
        ),
        'macro_f1': f1_score(
            y_true_labels,
            prediction_labels,
            average='macro',
        ),
        'mcc': matthews_corrcoef(
            y_true_labels,
            prediction_labels,
        ),
        'log_loss': log_loss(
            y_true_labels,
            probabilities,
            labels=class_order,
        ),
        'brier_multiclass': multiclass_brier(
            y_true_indices,
            probabilities,
        ),
        'ece_equal_width': expected_calibration_error(
            y_true_indices,
            probabilities,
            n_bins=N_ECE_BINS,
        ),
        'ece_equal_frequency': adaptive_ece(
            y_true_indices,
            probabilities,
            n_bins=N_ADAPTIVE_BINS,
        ),
        'mean_confidence': float(
            probabilities.max(axis=1).mean()
        ),
        'macro_roc_auc_ovr': roc_auc_score(
            y_binary,
            probabilities,
            average='macro',
            multi_class='ovr',
        ),
        'macro_average_precision': average_precision_score(
            y_binary,
            probabilities,
            average='macro',
        ),
        'prediction_labels': prediction_labels,
        'prediction_indices': prediction_indices,
        'y_true_indices': y_true_indices,
    }


def safe_name(value):
    return re.sub(
        r'[^A-Za-z0-9_.-]+',
        '_',
        str(value),
    )


## 7. Ajustar modelos y generar predicciones

In [ ]:

global_metric_rows = []
class_metric_rows = []
confusion_rows = []
temperature_rows = []
model_manifest_rows = []
prediction_manifest_rows = []
failure_rows = []

total_models = (
    len(DATASETS)
    * len(REPRESENTATIONS)
    * len(ALGORITHMS)
)
model_counter = 0

selected_lookup = (
    selected_configurations
    .set_index(
        ['dataset_id', 'representation', 'algorithm']
    )
)

for dataset_id in DATASETS:
    for representation in REPRESENTATIONS:
        dataframe = datasets[dataset_id][representation]
        features = feature_columns[dataset_id][representation]

        train_data = dataframe[
            dataframe['partition'] == 'train'
        ].copy()
        calibration_data = dataframe[
            dataframe['partition'] == 'calibration'
        ].copy()
        test_data = dataframe[
            dataframe['partition'] == 'test'
        ].copy()

        class_order = sorted(
            train_data['class_label'].astype(str).unique()
        )
        class_to_index = {
            class_label: index
            for index, class_label in enumerate(class_order)
        }

        x_train = train_data[features].to_numpy(dtype=float)
        y_train = train_data['class_label'].astype(str).to_numpy()

        x_calibration = calibration_data[
            features
        ].to_numpy(dtype=float)
        y_calibration = calibration_data[
            'class_label'
        ].astype(str).to_numpy()

        x_test = test_data[features].to_numpy(dtype=float)
        y_test = test_data[
            'class_label'
        ].astype(str).to_numpy()

        y_calibration_indices = np.array(
            [
                class_to_index[label]
                for label in y_calibration
            ],
            dtype=int,
        )

        for algorithm in ALGORITHMS:
            model_counter += 1

            selected = selected_lookup.loc[
                (dataset_id, representation, algorithm)
            ]
            parameters = json.loads(
                selected['parameters_json']
            )

            print(
                f'[{model_counter}/{total_models}]',
                dataset_id,
                representation,
                algorithm,
                parameters,
            )

            start_time = time.perf_counter()

            try:
                estimator = build_estimator(
                    algorithm,
                    parameters,
                    RANDOM_SEED,
                )
                estimator.fit(x_train, y_train)

                native_calibration_probabilities = (
                    aligned_probabilities(
                        estimator,
                        x_calibration,
                        class_order,
                    )
                )
                native_test_probabilities = (
                    aligned_probabilities(
                        estimator,
                        x_test,
                        class_order,
                    )
                )

                temperature, calibration_nll_after = (
                    fit_temperature(
                        native_calibration_probabilities,
                        y_calibration_indices,
                    )
                )

                calibrated_calibration_probabilities = (
                    probability_temperature_scale(
                        native_calibration_probabilities,
                        temperature,
                    )
                )
                calibrated_test_probabilities = (
                    probability_temperature_scale(
                        native_test_probabilities,
                        temperature,
                    )
                )

                calibration_native_metrics = (
                    compute_global_metrics(
                        y_calibration,
                        native_calibration_probabilities,
                        class_order,
                    )
                )
                calibration_calibrated_metrics = (
                    compute_global_metrics(
                        y_calibration,
                        calibrated_calibration_probabilities,
                        class_order,
                    )
                )

                test_native_metrics = compute_global_metrics(
                    y_test,
                    native_test_probabilities,
                    class_order,
                )
                test_calibrated_metrics = compute_global_metrics(
                    y_test,
                    calibrated_test_probabilities,
                    class_order,
                )

                elapsed_seconds = (
                    time.perf_counter() - start_time
                )

                model_output_dir = (
                    MODEL_DIR / dataset_id / representation
                )
                model_output_dir.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                model_path = (
                    model_output_dir
                    / f'{algorithm}.joblib'
                )
                joblib.dump(
                    estimator,
                    model_path,
                    compress=3,
                )

                metadata = {
                    'dataset_id': dataset_id,
                    'representation': representation,
                    'algorithm': algorithm,
                    'candidate_id': selected['candidate_id'],
                    'parameters': parameters,
                    'random_seed': RANDOM_SEED,
                    'feature_columns': features,
                    'class_order': class_order,
                    'temperature': temperature,
                    'train_records': len(train_data),
                    'calibration_records': len(calibration_data),
                    'test_records': len(test_data),
                    'fit_and_evaluation_seconds': elapsed_seconds,
                }

                metadata_path = (
                    model_output_dir
                    / f'{algorithm}_metadata.json'
                )
                metadata_path.write_text(
                    json.dumps(
                        metadata,
                        indent=2,
                        ensure_ascii=False,
                    ),
                    encoding='utf-8',
                )

                model_manifest_rows.append({
                    **{
                        key: metadata[key]
                        for key in [
                            'dataset_id',
                            'representation',
                            'algorithm',
                            'candidate_id',
                            'temperature',
                            'train_records',
                            'calibration_records',
                            'test_records',
                            'fit_and_evaluation_seconds',
                        ]
                    },
                    'feature_count': len(features),
                    'class_count': len(class_order),
                    'model_path': str(model_path),
                    'model_size_bytes': model_path.stat().st_size,
                    'metadata_path': str(metadata_path),
                })

                temperature_rows.append({
                    'dataset_id': dataset_id,
                    'representation': representation,
                    'algorithm': algorithm,
                    'temperature': temperature,
                    'calibration_native_log_loss': (
                        calibration_native_metrics['log_loss']
                    ),
                    'calibration_calibrated_log_loss': (
                        calibration_calibrated_metrics['log_loss']
                    ),
                    'calibration_log_loss_change': (
                        calibration_calibrated_metrics['log_loss']
                        - calibration_native_metrics['log_loss']
                    ),
                    'calibration_native_ece': (
                        calibration_native_metrics[
                            'ece_equal_width'
                        ]
                    ),
                    'calibration_calibrated_ece': (
                        calibration_calibrated_metrics[
                            'ece_equal_width'
                        ]
                    ),
                    'optimizer_nll': calibration_nll_after,
                })

                for partition_name, partition_data, y_labels, native_probabilities, calibrated_probabilities in [
                    (
                        'calibration',
                        calibration_data,
                        y_calibration,
                        native_calibration_probabilities,
                        calibrated_calibration_probabilities,
                    ),
                    (
                        'test',
                        test_data,
                        y_test,
                        native_test_probabilities,
                        calibrated_test_probabilities,
                    ),
                ]:
                    prediction_output_dir = (
                        PREDICTION_DIR
                        / dataset_id
                        / representation
                    )
                    prediction_output_dir.mkdir(
                        parents=True,
                        exist_ok=True,
                    )

                    native_prediction_indices = (
                        native_probabilities.argmax(axis=1)
                    )
                    calibrated_prediction_indices = (
                        calibrated_probabilities.argmax(axis=1)
                    )

                    prediction_frame = partition_data[
                        [
                            'record_id',
                            'duplicate_group_id',
                            'class_label',
                            'partition',
                        ]
                    ].copy()

                    prediction_frame['predicted_native'] = (
                        np.asarray(class_order)[
                            native_prediction_indices
                        ]
                    )
                    prediction_frame['predicted_calibrated'] = (
                        np.asarray(class_order)[
                            calibrated_prediction_indices
                        ]
                    )
                    prediction_frame['confidence_native'] = (
                        native_probabilities.max(axis=1)
                    )
                    prediction_frame['confidence_calibrated'] = (
                        calibrated_probabilities.max(axis=1)
                    )
                    prediction_frame['correct_native'] = (
                        prediction_frame['predicted_native']
                        == prediction_frame['class_label'].astype(str)
                    )
                    prediction_frame['correct_calibrated'] = (
                        prediction_frame['predicted_calibrated']
                        == prediction_frame['class_label'].astype(str)
                    )

                    for class_index, class_label in enumerate(class_order):
                        prediction_frame[
                            f'prob_native_{class_index}'
                        ] = native_probabilities[:, class_index]
                        prediction_frame[
                            f'prob_calibrated_{class_index}'
                        ] = calibrated_probabilities[:, class_index]

                    prediction_path = (
                        prediction_output_dir
                        / f'{algorithm}_{partition_name}_predictions.parquet'
                    )
                    prediction_frame.to_parquet(
                        prediction_path,
                        index=False,
                    )

                    class_map_path = (
                        prediction_output_dir
                        / f'{algorithm}_class_map.json'
                    )
                    class_map_path.write_text(
                        json.dumps(
                            {
                                str(index): class_label
                                for index, class_label
                                in enumerate(class_order)
                            },
                            indent=2,
                            ensure_ascii=False,
                        ),
                        encoding='utf-8',
                    )

                    prediction_manifest_rows.append({
                        'dataset_id': dataset_id,
                        'representation': representation,
                        'algorithm': algorithm,
                        'partition': partition_name,
                        'rows': len(prediction_frame),
                        'prediction_path': str(prediction_path),
                        'prediction_size_bytes': (
                            prediction_path.stat().st_size
                        ),
                        'class_map_path': str(class_map_path),
                    })

                for probability_status, metrics in [
                    ('native', test_native_metrics),
                    ('temperature_scaled', test_calibrated_metrics),
                ]:
                    global_metric_rows.append({
                        'dataset_id': dataset_id,
                        'representation': representation,
                        'algorithm': algorithm,
                        'probability_status': probability_status,
                        'scale_factor': 1.0,
                        'n_test': len(test_data),
                        'accuracy': metrics['accuracy'],
                        'balanced_accuracy': (
                            metrics['balanced_accuracy']
                        ),
                        'macro_f1': metrics['macro_f1'],
                        'mcc': metrics['mcc'],
                        'macro_roc_auc_ovr': (
                            metrics['macro_roc_auc_ovr']
                        ),
                        'macro_average_precision': (
                            metrics['macro_average_precision']
                        ),
                        'log_loss': metrics['log_loss'],
                        'brier_multiclass': (
                            metrics['brier_multiclass']
                        ),
                        'ece_equal_width': (
                            metrics['ece_equal_width']
                        ),
                        'ece_equal_frequency': (
                            metrics['ece_equal_frequency']
                        ),
                        'mean_confidence': (
                            metrics['mean_confidence']
                        ),
                    })

                precision, recall, class_f1, support = (
                    precision_recall_fscore_support(
                        y_test,
                        test_native_metrics[
                            'prediction_labels'
                        ],
                        labels=class_order,
                        zero_division=0,
                    )
                )

                for class_index, class_label in enumerate(class_order):
                    class_metric_rows.append({
                        'dataset_id': dataset_id,
                        'representation': representation,
                        'algorithm': algorithm,
                        'scale_factor': 1.0,
                        'class_index': class_index,
                        'class_label': class_label,
                        'support': int(support[class_index]),
                        'precision': float(precision[class_index]),
                        'recall': float(recall[class_index]),
                        'f1': float(class_f1[class_index]),
                    })

                matrix = confusion_matrix(
                    y_test,
                    test_native_metrics[
                        'prediction_labels'
                    ],
                    labels=class_order,
                )
                normalized_matrix = confusion_matrix(
                    y_test,
                    test_native_metrics[
                        'prediction_labels'
                    ],
                    labels=class_order,
                    normalize='true',
                )

                for true_index, true_class in enumerate(class_order):
                    for predicted_index, predicted_class in enumerate(
                        class_order
                    ):
                        confusion_rows.append({
                            'dataset_id': dataset_id,
                            'representation': representation,
                            'algorithm': algorithm,
                            'scale_factor': 1.0,
                            'true_class': true_class,
                            'predicted_class': predicted_class,
                            'count': int(
                                matrix[true_index, predicted_index]
                            ),
                            'row_normalized_proportion': float(
                                normalized_matrix[
                                    true_index,
                                    predicted_index,
                                ]
                            ),
                        })

            except Exception as error:
                print(
                    'ERROR:',
                    dataset_id,
                    representation,
                    algorithm,
                    type(error).__name__,
                    error,
                )
                traceback.print_exc()

                failure_rows.append({
                    'dataset_id': dataset_id,
                    'representation': representation,
                    'algorithm': algorithm,
                    'error_type': type(error).__name__,
                    'error_message': str(error),
                })

if failure_rows:
    failures = pd.DataFrame(failure_rows)
    failures.to_csv(
        RESULTS_DIR / 'FAILURES.csv',
        index=False,
        encoding='utf-8-sig',
    )
    raise RuntimeError(
        f'Fallaron {len(failure_rows)} modelos. '
        'Revisa FAILURES.csv.'
    )

print('Entrenamiento, calibración y predicción completados para 32 modelos.')


## 8. Guardar resultados tabulares

In [ ]:

global_metrics = pd.DataFrame(global_metric_rows)
class_metrics = pd.DataFrame(class_metric_rows)
confusion_long = pd.DataFrame(confusion_rows)
temperatures = pd.DataFrame(temperature_rows)
model_manifest = pd.DataFrame(model_manifest_rows)
prediction_manifest = pd.DataFrame(prediction_manifest_rows)

global_metrics.to_csv(
    RESULTS_DIR / '01_baseline_test_global_metrics.csv',
    index=False,
    encoding='utf-8-sig',
)
class_metrics.to_csv(
    RESULTS_DIR / '02_baseline_test_class_metrics.csv',
    index=False,
    encoding='utf-8-sig',
)
confusion_long.to_csv(
    RESULTS_DIR / '03_baseline_test_confusion_matrices_long.csv',
    index=False,
    encoding='utf-8-sig',
)
temperatures.to_csv(
    RESULTS_DIR / '04_temperature_scaling_parameters.csv',
    index=False,
    encoding='utf-8-sig',
)
model_manifest.to_csv(
    RESULTS_DIR / '05_final_model_manifest.csv',
    index=False,
    encoding='utf-8-sig',
)
prediction_manifest.to_csv(
    RESULTS_DIR / '06_prediction_file_manifest.csv',
    index=False,
    encoding='utf-8-sig',
)

native_metrics = global_metrics[
    global_metrics['probability_status'] == 'native'
].copy()

calibrated_metrics = global_metrics[
    global_metrics['probability_status']
    == 'temperature_scaled'
].copy()

representation_summary = (
    native_metrics.groupby(
        ['dataset_id', 'representation']
    )
    .agg(
        algorithms=('algorithm', 'nunique'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_sd=('macro_f1', 'std'),
        macro_f1_min=('macro_f1', 'min'),
        macro_f1_max=('macro_f1', 'max'),
        balanced_accuracy_mean=(
            'balanced_accuracy',
            'mean',
        ),
        mcc_mean=('mcc', 'mean'),
        macro_roc_auc_mean=(
            'macro_roc_auc_ovr',
            'mean',
        ),
        macro_ap_mean=(
            'macro_average_precision',
            'mean',
        ),
    )
    .reset_index()
)

representation_summary.to_csv(
    TABLES_DIR / 'Table_baseline_representation_summary.csv',
    index=False,
    encoding='utf-8-sig',
)

calibration_comparison = (
    global_metrics.pivot_table(
        index=[
            'dataset_id',
            'representation',
            'algorithm',
        ],
        columns='probability_status',
        values=[
            'log_loss',
            'brier_multiclass',
            'ece_equal_width',
            'ece_equal_frequency',
            'mean_confidence',
        ],
    )
)

calibration_comparison.columns = [
    f'{metric}_{status}'
    for metric, status in calibration_comparison.columns
]
calibration_comparison = (
    calibration_comparison.reset_index()
)

for metric in [
    'log_loss',
    'brier_multiclass',
    'ece_equal_width',
    'ece_equal_frequency',
]:
    calibration_comparison[
        f'{metric}_change_calibrated_minus_native'
    ] = (
        calibration_comparison[
            f'{metric}_temperature_scaled'
        ]
        - calibration_comparison[
            f'{metric}_native'
        ]
    )

calibration_comparison.to_csv(
    TABLES_DIR / 'Table_baseline_calibration_comparison.csv',
    index=False,
    encoding='utf-8-sig',
)

display(representation_summary)
display(calibration_comparison.head(20))


## 9. Figuras de evaluación base

In [ ]:

for dataset_id in DATASETS:
    plot_data = native_metrics[
        native_metrics['dataset_id'] == dataset_id
    ].copy()

    matrix = (
        plot_data.pivot(
            index='algorithm',
            columns='representation',
            values='macro_f1',
        )
        .reindex(
            index=ALGORITHMS,
            columns=REPRESENTATIONS,
        )
    )

    figure, axis = plt.subplots(figsize=(10, 6))
    image = axis.imshow(
        matrix.to_numpy(),
        aspect='auto',
    )

    axis.set_xticks(
        np.arange(len(REPRESENTATIONS))
    )
    axis.set_xticklabels(
        REPRESENTATIONS,
        rotation=35,
        ha='right',
    )
    axis.set_yticks(
        np.arange(len(ALGORITHMS))
    )
    axis.set_yticklabels(ALGORITHMS)
    axis.set_xlabel('Representación')
    axis.set_ylabel('Algoritmo')
    axis.set_title(
        f'Macro-F1 de prueba en escala base — {dataset_id}'
    )

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            axis.text(
                column_index,
                row_index,
                f'{matrix.iloc[row_index, column_index]:.3f}',
                ha='center',
                va='center',
                fontsize=9,
            )

    figure.colorbar(
        image,
        ax=axis,
        label='Macro-F1',
    )
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f'baseline_macro_f1_{dataset_id}.png',
        dpi=300,
        bbox_inches='tight',
    )
    figure.savefig(
        FIGURES_DIR / f'baseline_macro_f1_{dataset_id}.pdf',
        bbox_inches='tight',
    )
    plt.show()

    calibration_plot = calibration_comparison[
        calibration_comparison['dataset_id'] == dataset_id
    ].copy()

    x = np.arange(len(calibration_plot))
    figure, axis = plt.subplots(figsize=(13, 6))

    axis.scatter(
        x,
        calibration_plot['log_loss_native'],
        label='Native',
        marker='o',
    )
    axis.scatter(
        x,
        calibration_plot[
            'log_loss_temperature_scaled'
        ],
        label='Temperature-scaled',
        marker='s',
    )

    for index, row in calibration_plot.reset_index(drop=True).iterrows():
        axis.plot(
            [
                index,
                index,
            ],
            [
                row['log_loss_native'],
                row['log_loss_temperature_scaled'],
            ],
            linewidth=1,
        )

    labels = (
        calibration_plot['representation']
        + ' | '
        + calibration_plot['algorithm']
    )

    axis.set_xticks(x)
    axis.set_xticklabels(
        labels,
        rotation=75,
        ha='right',
        fontsize=8,
    )
    axis.set_ylabel('Log loss en prueba')
    axis.set_title(
        f'Calibración por temperatura — {dataset_id}'
    )
    axis.legend()
    axis.grid(True, axis='y', alpha=0.3)
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f'baseline_temperature_logloss_{dataset_id}.png',
        dpi=300,
        bbox_inches='tight',
    )
    figure.savefig(
        FIGURES_DIR / f'baseline_temperature_logloss_{dataset_id}.pdf',
        bbox_inches='tight',
    )
    plt.show()


## 10. Auditoría metodológica y validación final

In [ ]:

expected_models = (
    len(DATASETS)
    * len(REPRESENTATIONS)
    * len(ALGORITHMS)
)

native_key = (
    native_metrics[
        [
            'dataset_id',
            'representation',
            'algorithm',
            'macro_f1',
            'accuracy',
            'balanced_accuracy',
            'mcc',
        ]
    ]
    .sort_values(
        ['dataset_id', 'representation', 'algorithm']
    )
    .reset_index(drop=True)
)

calibrated_key = (
    calibrated_metrics[
        [
            'dataset_id',
            'representation',
            'algorithm',
            'macro_f1',
            'accuracy',
            'balanced_accuracy',
            'mcc',
        ]
    ]
    .sort_values(
        ['dataset_id', 'representation', 'algorithm']
    )
    .reset_index(drop=True)
)

discrimination_unchanged = np.allclose(
    native_key[
        [
            'macro_f1',
            'accuracy',
            'balanced_accuracy',
            'mcc',
        ]
    ].to_numpy(),
    calibrated_key[
        [
            'macro_f1',
            'accuracy',
            'balanced_accuracy',
            'mcc',
        ]
    ].to_numpy(),
    atol=1e-12,
    rtol=0.0,
)

calibration_nll_not_worse = bool(
    (
        temperatures[
            'calibration_calibrated_log_loss'
        ]
        <= (
            temperatures[
                'calibration_native_log_loss'
            ]
            + 1e-9
        )
    ).all()
)

prediction_files_exist = all(
    Path(path).exists()
    for path in prediction_manifest['prediction_path']
)

model_files_exist = all(
    Path(path).exists()
    for path in model_manifest['model_path']
)

final_checks = {
    'notebook03_and_04_validated': True,
    'exactly_32_models_fitted': (
        len(model_manifest) == expected_models
    ),
    'exactly_32_temperature_parameters': (
        len(temperatures) == expected_models
    ),
    'native_and_calibrated_metrics_complete': (
        len(global_metrics) == expected_models * 2
    ),
    'no_missing_global_metrics': bool(
        global_metrics.select_dtypes(
            include=[np.number]
        ).notna().all(axis=None)
    ),
    'temperature_positive': bool(
        (temperatures['temperature'] > 0).all()
    ),
    'calibration_nll_not_worse_on_calibration_set': (
        calibration_nll_not_worse
    ),
    'temperature_scaling_preserves_discrimination': (
        discrimination_unchanged
    ),
    'all_model_files_exist': model_files_exist,
    'all_prediction_files_exist': prediction_files_exist,
    'test_not_used_for_temperature_fitting': True,
}

final_validation = pd.DataFrame([
    {
        'check': check,
        'passed': passed,
    }
    for check, passed in final_checks.items()
])

final_validation.to_csv(
    RESULTS_DIR / '07_final_baseline_validation.csv',
    index=False,
    encoding='utf-8-sig',
)

if not final_validation['passed'].all():
    raise RuntimeError(
        'Falló al menos una validación final.'
    )

print('Validación final de evaluación base: CORRECTA')
display(final_validation)


## 11. Manifiesto de integridad SHA-256

In [ ]:

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


integrity_rows = []

for path in OUTPUT_DIR.rglob('*'):
    if path.is_file():
        integrity_rows.append({
            'relative_path': str(
                path.relative_to(OUTPUT_DIR)
            ),
            'size_bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        })

integrity_manifest = pd.DataFrame(
    integrity_rows
).sort_values('relative_path')

integrity_manifest.to_csv(
    RESULTS_DIR / '08_output_integrity_manifest.csv',
    index=False,
    encoding='utf-8-sig',
)

display(integrity_manifest.head(30))


## 12. Congelar protocolo de evaluación base

In [ ]:

baseline_protocol = {
    'protocol_name': (
        'Grain_Robustness_Q1_baseline_calibration_protocol_v1'
    ),
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'random_seed': RANDOM_SEED,
    'datasets': DATASETS,
    'representations': REPRESENTATIONS,
    'algorithms': ALGORITHMS,
    'selected_configuration_source': (
        'NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip'
    ),
    'training_partition': 'train',
    'temperature_fitting_partition': 'calibration',
    'evaluation_partition': 'test',
    'baseline_scale_factor': 1.0,
    'probability_calibration': {
        'method': (
            'single scalar temperature applied to log native '
            'probability vectors'
        ),
        'optimization_target': (
            'multiclass negative log-likelihood on calibration'
        ),
        'temperature_bounds': [0.05, 10.0],
        'argmax_preserved': True,
    },
    'global_metrics': [
        'accuracy',
        'balanced_accuracy',
        'macro_f1',
        'matthews_correlation_coefficient',
        'macro_roc_auc_ovr',
        'macro_average_precision',
        'multiclass_log_loss',
        'multiclass_brier_score',
        'ece_equal_width_15_bins',
        'ece_equal_frequency_10_bins',
        'mean_confidence',
    ],
    'restrictions': [
        'No hyperparameter reselection.',
        'No representation selection from test results.',
        'No temperature fitting on test.',
        'No claim of real camera-domain validation.',
        'Baseline results are part of the prespecified final test matrix.',
    ],
    'hardware': {
        'recommended': 'Google Colab CPU',
        'gpu_required': False,
    },
}

protocol_path = (
    PROTOCOL_DIR
    / 'baseline_calibration_protocol_v1.json'
)
protocol_path.write_text(
    json.dumps(
        baseline_protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

print(json.dumps(baseline_protocol, indent=2, ensure_ascii=False))


## 13. Crear ZIP y copiar a Drive

In [ ]:

zip_local = (
    LOCAL_ROOT
    / 'NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip'
)

if zip_local.exists():
    zip_local.unlink()

with zipfile.ZipFile(
    zip_local,
    'w',
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_DIR),
            )


def copy_with_retry(source, destination, attempts=8):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            shutil.copy2(source, destination)

            if (
                not destination.exists()
                or destination.stat().st_size
                != source.stat().st_size
            ):
                raise IOError(
                    'La copia no pudo verificarse.'
                )

            return True

        except Exception as error:
            last_error = error
            print(
                f'Intento {attempt}/{attempts}: '
                f'{type(error).__name__}: {error}'
            )
            time.sleep(5)

    print(
        'No se pudo copiar a Drive. '
        f'Último error: {last_error}'
    )
    return False


zip_drive = (
    DRIVE_PROJECT_ROOT / '07_results'
    / 'NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip'
)

copied = copy_with_retry(
    zip_local,
    zip_drive,
)

print('Notebook 05 completado correctamente.')
print('ZIP local:', zip_local)
print('ZIP en Drive:', zip_drive)
print('Tamaño ZIP MB:', round(zip_local.stat().st_size / 1_000_000, 3))
print('Copia en Drive verificada:', copied)

if not copied:
    print(
        'Descarga manualmente el ZIP local desde el panel Archivos de Colab.'
    )



## Archivo que debes compartir

Al finalizar, envía:

```text
/content/drive/MyDrive/Grain_Robustness_Q1/07_results/
NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip
```

Si Drive falla:

```text
/content/Grain_Robustness_Q1_notebook05/
NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip
```

El notebook debe mostrar:

```text
Entradas de los Notebooks 03 y 04 validadas correctamente.
Entrenamiento, calibración y predicción completados para 32 modelos.
Validación final de evaluación base: CORRECTA
Notebook 05 completado correctamente.
```
